In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, roc_curve

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.inspection import permutation_importance

from sklearn.utils.class_weight import compute_class_weight

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, train_test_split

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.feature_extraction.text import TfidfVectorizer
# import nltk
# from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
df = pd.read_csv("depression-classification-text-dataset.csv").dropna()

In [3]:
df

,text,label
0,"I'm feeling really down today, like everything...",1.0
1,"Just had a great workout, feeling fantastic an...",0.0
2,I can't seem to shake this feeling of emptines...,1.0
3,"Spent the day with friends, laughing and enjoy...",0.0
4,"I don't see the point in anything anymore, it ...",1.0
...,...,...
1583,Lost in a maze of endless possibilities.,1.0
1584,Seeking refuge in the sanctuary of solitude.,0.0
1585,Lost in the wilderness of my own mind.,1.0
1586,Finding strength in the face of adversity.,0.0


In [4]:
X = df.drop(columns=["label"])

In [5]:
y = df["label"]

In [6]:
# Assume X and y represent the full dataset (15000 samples total)
X_original = X[:1200]  # First 1200 examples
y_original = y[:1200]

X_extra = X[1200:]     # Additional 300 examples
y_extra = y[1200:]

In [7]:
X_original

,text
0,"I'm feeling really down today, like everything..."
1,"Just had a great workout, feeling fantastic an..."
2,I can't seem to shake this feeling of emptines...
3,"Spent the day with friends, laughing and enjoy..."
4,"I don't see the point in anything anymore, it ..."
...,...
1196,I feel like I'm losing the battle against my o...
1197,I'm trying to find the light at the end of the...
1198,Enjoying the serenity of a peaceful morning.
1199,"I feel like I'm running on empty, with no fuel..."


In [8]:
# Split the original data for validation
X_train_orig, X_val_orig, y_train_orig, y_val_orig = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42, stratify=y_original)

In [9]:
X_train_combined = np.concatenate([X_train_orig, X_extra])
y_train_combined = np.concatenate([y_train_orig, y_extra])

In [10]:
X_train_combined

array([["I'm exhausted from trying to keep it together."],
       ["I don't know how much longer I can keep going like this."],
       ['Taking a break to enjoy nature, feeling refreshed.'],
       ...,
       ['Lost in the wilderness of my own mind.'],
       ['Finding strength in the face of adversity.'],
       ['Adrift in a sea of uncertainty and doubt.']], dtype=object)

In [11]:
y_train_combined

array([1., 1., 0., ..., 1., 0., 1.])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    # ('feature_selection', SelectKBest(mutual_info_classif, k=10)),
    ('classifier', CalibratedClassifierCV(LinearSVC(max_iter=10000), cv=skf, method="sigmoid"))
])
# Grid search over LinearSVC parameters (use model__estimator__ to access nested parameters)
param_grid = {
    'classifier__estimator__C': [0.001, 0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
)
grid.fit(X_train_combined, y_train_combined)

# Results
print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation Score:", grid.best_score_)
# print("Test Accuracy:", grid.score(X_test, y_test))

In [ ]:
result = permutation_importance(grid.best_estimator_, X_val, y_val, n_repeats=10)

In [ ]:
result

In [ ]:
# Predict
y_pred = grid.best_estimator_.predict(X_val)

In [ ]:
y_pred

In [ ]:
y_probs = grid.best_estimator_.predict_proba(X_val)[:, 1]

In [ ]:
y_probs

In [ ]:
thresholds = np.linspace(0, 1, 100)
best_threshold = 0.5
best_f1 = 0

# Store metrics
results = {
    'Threshold': [],
    'Precision': [],
    'Recall': [],
    'F1 Score': []
}

f1_scores = []

for threshold in thresholds:
    y_pred = (y_probs >= threshold).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
    f1_scores.append(f1)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    results['Threshold'].append(threshold)
    results['Precision'].append(precision)
    results['Recall'].append(recall)
    results['F1 Score'].append(f1)

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1 score: {best_f1:.4f}")

In [ ]:
df_results = pd.DataFrame(results)
print(df_results.sort_values('F1 Score', ascending=False).head())

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)

print(f"Best threshold: {thresholds[best_idx]:.2f}")
print(f"Best F1 score: {f1_scores[best_idx]:.4f}")

In [ ]:
# # Plot
# plt.figure(figsize=(8, 5))
# plt.plot(thresholds, f1_scores, label='F1 Score')
# plt.xlabel('Threshold')
# plt.ylabel('F1 Score')
# plt.title('F1 Score vs. Classification Threshold')
# plt.grid(True)
# plt.legend()
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))
# plt.plot(df_results['Threshold'], df_results['Precision'], label='Precision')
# plt.plot(df_results['Threshold'], df_results['Recall'], label='Recall')
# plt.plot(df_results['Threshold'], df_results['F1 Score'], label='F1 Score')
# plt.xlabel('Threshold')
# plt.ylabel('Metric Score')
# plt.title('Precision, Recall, F1 vs. Threshold')
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()